# EDA — Importaciones de Comercio Exterior (INE Bolivia)

Análisis exploratorio de **un archivo individual** de importaciones del INE.

**Uso:** Este notebook recibe un parámetro `FILE_PATH` con la ruta al archivo `.xlsx`.
Se puede ejecutar directamente o mediante el notebook orquestador con `papermill`.

**Nota:** Los archivos de importaciones son significativamente más grandes (~400k filas, ~60 MB).
Se puede limitar la carga con el parámetro `MAX_ROWS`.

In [ ]:
# Parámetros de entrada — inyectados por papermill
FILE_PATH = r"data/raw/comercio exterior/importaciones/IMPORTACIONES_2021.xlsx"
MAX_ROWS = None  # None = leer todo; usar un número para limitar (ej: 50000)

## 1. Configuración e Importaciones

In [ ]:
import sys
from pathlib import Path

import pandas as pd

# Asegurar que src/ es importable resolviendo la raíz del proyecto
current = Path.cwd().resolve()
candidates = [
    current,
    current.parent,
    current / "insight-bolivia",
    current.parent / "insight-bolivia",
    *current.parents,
]
PROJECT_ROOT = next(
    (p for p in candidates if (p / "src" / "extract.py").exists()),
    current,
)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.extract import get_excel_metadata, read_ine_excel
from src.transform import clean_import_dataframe, compute_null_report
from src.validate import run_import_validations

pd.set_option("display.max_columns", 50)
pd.set_option("display.max_colwidth", 80)

filepath = Path(FILE_PATH)
if not filepath.is_absolute():
    filepath = PROJECT_ROOT / filepath
print(f"Archivo: {filepath.name}")
print(f"MAX_ROWS: {MAX_ROWS if MAX_ROWS else 'Sin límite'}")
print(f"Ruta completa: {filepath.resolve()}")

## 2. Metadatos del Archivo

In [ ]:
meta = get_excel_metadata(filepath)

print(f"Archivo:       {meta['filename']}")
print(f"Tamaño:        {meta['file_size_mb']} MB")
print(f"Hojas:         {meta['sheet_names']}")
print(f"Hoja activa:   {meta['active_sheet']}")
print(f"Columnas ({meta['n_columns']}):")
for i, col in enumerate(meta['headers'], 1):
    print(f"  {i:2d}. {col}")

## 3. Carga y Limpieza de Datos

In [ ]:
df_raw = read_ine_excel(filepath, max_rows=MAX_ROWS)
print(f"Shape crudo: {df_raw.shape}")
df_raw.head(3)

In [ ]:
df = clean_import_dataframe(df_raw)
print(f"Shape limpio: {df.shape}")
df.dtypes

## 4. Análisis del Código NANDINA

In [ ]:
nandina = df["NANDINA"]
print(f"Tipo de dato: {nandina.dtype}")
print(f"Valores únicos: {nandina.nunique()}")
print("Longitud (value_counts):")
print(nandina.str.len().value_counts().sort_index())
print(f"\nRegistros con cero a la izquierda: {(nandina.str[0] == '0').sum()}")
print(f"Porcentaje: {(nandina.str[0] == '0').mean() * 100:.1f}%")

## 5. Análisis de Valores Monetarios y Tipo de Cambio

In [ ]:
value_cols = ["FOB", "FRO", "ADU", "PAG"]
existing = [c for c in value_cols if c in df.columns]
print("Estadísticas de columnas monetarias:")
df[existing].describe().round(2)

In [ ]:
# Verificar tipo de cambio BOB/USD (esperado: 6.96)
if "ADU" in df.columns and "FRO" in df.columns:
    mask = df["FRO"].notna() & df["ADU"].notna() & (df["FRO"] > 0)
    if mask.sum() > 0:
        ratio = df.loc[mask, "ADU"] / df.loc[mask, "FRO"]
        print("Tipo de cambio ADU/FRO (CIF BOB / CIF USD):")
        print(f"  Media:    {ratio.mean():.4f}")
        print(f"  Mediana:  {ratio.median():.4f}")
        print(f"  Min:      {ratio.min():.4f}")
        print(f"  Max:      {ratio.max():.4f}")
        print("  Esperado: 6.96")
        outliers = ratio[(ratio < 6.86) | (ratio > 7.06)]
        print(f"  Outliers (fuera de 6.86-7.06): {len(outliers)} de {mask.sum()}")

## 6. Análisis de Peso (KILOS)

In [ ]:
if "KILOS" in df.columns:
    kilos = df["KILOS"]
    print("Estadísticas de KILOS (peso bruto):")
    print(kilos.describe().round(2))
    print(f"\nRegistros con KILOS = 0: {(kilos == 0).sum()}")
    print(f"Registros con KILOS < 0: {(kilos < 0).sum()}")

## 7. Reporte de Nulos

In [ ]:
null_report = compute_null_report(df)
with_nulls = null_report[null_report["nulos"] > 0]
if with_nulls.empty:
    print("¡Sin nulos!")
else:
    print("Columnas con nulos:")
    print(with_nulls.to_string(index=False))

## 8. Validaciones de Calidad

In [ ]:
results = run_import_validations(df)
print("Resultados de validación:")
print("-" * 60)
for r in results:
    status = "✅ PASS" if r.passed else "❌ FAIL"
    print(f"{status} | {r.rule_name}: {r.message}")
    if r.details:
        for k, v in r.details.items():
            print(f"         {k}: {v}")
    print()

## 9. Distribución por País, Aduana y Departamento

In [ ]:
if "DESPAI" in df.columns:
    print("Top 15 países origen (por número de registros):")
    print(df["DESPAI"].value_counts().head(15))


if "DESADU" in df.columns:
    print("\nRegistros por aduana de ingreso:")
    print(df["DESADU"].value_counts())

if "DESDEPTO" in df.columns:
    print("\nRegistros por departamento:")
    print(df["DESDEPTO"].value_counts())

## 10. Resumen del Archivo

In [ ]:
print("=" * 60)
print(f"RESUMEN: {filepath.name}")
print("=" * 60)
rows_label = f"{len(df):,}" + (" (limitado)" if MAX_ROWS else "")
print(f"  Filas:              {rows_label}")
print(f"  Columnas:           {len(df.columns)}")
if "GESTION" in df.columns:
    print(f"  Gestión(es):        {sorted(df['GESTION'].dropna().unique())}")
if "MES" in df.columns:
    print(f"  Meses:              {sorted(df['MES'].dropna().unique())}")
if "NANDINA" in df.columns:
    print(f"  Productos únicos:   {df['NANDINA'].nunique():,}")
if "DESPAI" in df.columns:
    print(f"  Países origen:      {df['DESPAI'].nunique()}")
if "FOB" in df.columns:
    print(f"  Valor FOB total:    USD {df['FOB'].sum():,.2f}")
if "FRO" in df.columns:
    print(f"  Valor CIF total:    USD {df['FRO'].sum():,.2f}")
print(f"  Validaciones:       {sum(1 for r in results if r.passed)}/{len(results)} pasaron")